In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install openai
!pip install python-dotenv

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

In [ ]:
!pip install transformers
!pip install torch
!pip install huggingface_hub
!pip install evaluate
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


# GPT data collection and checking the performance on the data with fine-tuned roberta

# dataset install making question for gpt input

In [ ]:
from datasets import load_dataset

# Load a custom CSV file
data_files = {"train": "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/full_data.csv"}
dataset = load_dataset("csv", data_files=data_files)

In [ ]:
# dataset into pandas dataframe
import pandas as pd
df = dataset['train'].to_pandas()

In [ ]:
# remove duplicate answers
df_check = df.drop_duplicates("human_answers")

In [ ]:
import nltk
from nltk.lm import Vocabulary
nltk.download('punkt_tab')# using nltk word tokenizer
# checking the number of tokens of the text and , if it is less than 4, remove the rows
def check_length(sentence):
    tokenized_sentence = nltk.tokenize.word_tokenize(str(sentence))
    return len(tokenized_sentence)
# remove duplicates and less than 4 tokens of texts.
def preprocess_data(df):
  df = df.drop_duplicates(subset = ["human_answers"])
  df['length'] = df["human_answers"].apply(check_length)
  df = df[df['length'] > 3]
  df = df[["question", "length", "source"]]
  return df
df_processed = preprocess_data(df)

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
/tmp/ipykernel_13609/1483159615.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['length'] = df["human_answers"].apply(check_length)


In [ ]:
# checking category distribution
def check_category_numbers(df):
  answers = []
  sources_list = ['reddit_eli5', 'open_qa', 'wiki_csai','medicine','finance']
  for source in sources_list:
    a = df[(df["source"] == source)].dropna()
    number = len(a)
    answers.append([source, number])
  return pd.DataFrame(answers, columns = [["source", "number"]])

check_category_numbers(df_processed)

,source,number
0,reddit_eli5,45965
1,open_qa,1170
2,wiki_csai,771
3,medicine,1244
4,finance,3932


In [ ]:
# making list to make the inputs for GPT-5.4-mini extraction
gpt_input = []
for question, length in zip(df_processed["question"], df_processed["length"]):
  gpt_input.append([question, length])

# GPT 5.4 answer extraction

In [ ]:
import os
from google.colab import userdata
from openai import OpenAI

# getting own openAI API key
api_key = userdata.get('OPEN')
os.environ['OPENAI_API_KEY'] = api_key
AI_generated = []
client = OpenAI()

i = 1
# for loop to extract GPT5.4-mini answers
for question, length in gpt_input:
  # manually setting system input and question input
  developer_input = 'answer the question by using only sentences not structured forms.'

  # using length because the length distribution of texts become close to human's answers
  question_input = question + ' Absolutely do not use more than ' + str(length - 4) + ' words'
  response = client.responses.create(
    model= "gpt-5.4-mini",
    input= [
        {
            'role': "developer",
            'content': developer_input,
        },
        {
            'role': "user",
            'content': question_input,
        },
    ],
  )
  #AI_generated list stores question of the data and answers from GPT5.4 - mini
  AI_generated.append([question, response.output_text])
  print(str(i) + ' times' )
  i = i+1


Streaming output truncated to the last 5000 lines.
7080 times
7081 times
7082 times
7083 times
7084 times
7085 times
7086 times
7087 times
7088 times
7089 times
7090 times
7091 times
7092 times
7093 times
7094 times
7095 times
7096 times
7097 times
7098 times
7099 times
7100 times
7101 times
7102 times
7103 times
7104 times
7105 times
7106 times
7107 times
7108 times
7109 times
7110 times
7111 times
7112 times
7113 times
7114 times
7115 times
7116 times
7117 times
7118 times
7119 times
7120 times
7121 times
7122 times
7123 times
7124 times
7125 times
7126 times
7127 times
7128 times
7129 times
7130 times
7131 times
7132 times
7133 times
7134 times
7135 times
7136 times
7137 times
7138 times
7139 times
7140 times
7141 times
7142 times
7143 times
7144 times
7145 times
7146 times
7147 times
7148 times
7149 times
7150 times
7151 times
7152 times
7153 times
7154 times
7155 times
7156 times
7157 times
7158 times
7159 times
7160 times
7161 times
7162 times
7163 times
7164 times
7165 times
716

In [ ]:
#into dataframe and save for future use
df_new_gpt = pd.DataFrame(AI_generated, columns = ['question','gpt_5_4_mini'])
df_new_gpt.to_csv('/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini.csv', index = True)

# creating gpt5.4-mini dataset from stored files into test, val, and train sets

In [ ]:
# getting all of the gpt answer file and unify them
recent_gpt_path_1 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini1.csv"
recent_gpt_path_2 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini2.csv"
recent_gpt_path_3 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini3.csv"
recent_gpt_path_4 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini4.csv"
recent_gpt_path_5 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini5.csv"
recent_gpt_path_6 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini6.csv"
recent_gpt_path_7 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini7.csv"
recent_gpt_path_8 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini8.csv"
recent_gpt_path_9 = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/gpt-5.4_mini9.csv"

# original dataset for creating new dataset of human and gpt5.4-mini generated dataset
test_path = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/test.csv"
val_path = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/val.csv"
train_path = "/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/train.csv"

In [ ]:
import pandas as pd
# all of the dataset into pandas dataframe
df_test = pd.read_csv(test_path)
df_val = pd.read_csv(val_path)
df_train = pd.read_csv(train_path)

df_recent_1 = pd.read_csv(recent_gpt_path_1)
df_recent_2 = pd.read_csv(recent_gpt_path_2)
df_recent_3 = pd.read_csv(recent_gpt_path_3)
df_recent_4 = pd.read_csv(recent_gpt_path_4)
df_recent_5 = pd.read_csv(recent_gpt_path_5)
df_recent_6 = pd.read_csv(recent_gpt_path_6)
df_recent_7 = pd.read_csv(recent_gpt_path_7)
df_recent_8 = pd.read_csv(recent_gpt_path_8)
df_recent_9 = pd.read_csv(recent_gpt_path_9)


In [ ]:
# unify all of the gpt5.4-mini data, adding label as 1, and rename the columns into text and label
new_data = pd.concat([df_recent_1,df_recent_2,df_recent_3,df_recent_4,df_recent_5, df_recent_6, df_recent_7, df_recent_8, df_recent_9])[['gpt_5_4_mini']]
df_recent = new_data.assign(label = 1)
df_recent = df_recent[['gpt_5_4_mini', 'label']].rename(columns= {'gpt_5_4_mini' : 'text'})

In [ ]:
# test set
test_data = pd.concat([df_recent.iloc[0:4596], df_recent.iloc[45965:46082], df_recent.iloc[47135:47212], df_recent.iloc[47906:48030], df_recent.iloc[49150:49543]])
test_data

,text,label
0,Because “#1 New York Times Best Seller” is not...,1
1,Because “New York Times #1 Best Seller” is not...,1
2,Because “#1 best seller” is often used for dif...,1
3,"We use road salt because it melts ice, making ...",1
4,We use salt on roads because it helps melt ice...,1
...,...,...
338,Not automatically. A 52-week high can mean str...,1
339,"In India, the interest component of a personal...",1
340,"Yes, generally lower interest rates can encour...",1
341,"It can be a bad idea, but it is not always a d...",1


In [ ]:
# validation set
val_data = pd.concat([df_recent.iloc[4596:9192], df_recent.iloc[46082:46199], df_recent.iloc[47212:47289], df_recent.iloc[48030:48154], df_recent.iloc[49543:49936]])
val_data

,text,label
4596,IQ tests are like a ruler for some kinds of th...,1
4597,IQ tests are kind of like a ruler for certain ...,1
4598,IQ tests are fairly good at measuring a person...,1
4599,A fever is your body making itself hotter to h...,1
4600,"A fever is your body’s way of fighting germs, ...",1
...,...,...
731,"Yes, you can usually use credit card points to...",1
732,Contractors can often recoup taxation-related ...,1
733,A good rule is to keep the car payment at or b...,1
734,You can use your current home’s equity as part...,1


In [ ]:
# train set
train_data = pd.concat([df_recent.iloc[9192:45965], df_recent.iloc[46199:47135], df_recent.iloc[47289:47906], df_recent.iloc[48154:49150], df_recent.iloc[49936:53082]])
train_data

,text,label
9192,Guys often wake up with an erection because du...,1
9193,Morning erections happen because sleep changes...,1
9194,Guys often wake up with an erection because th...,1
9195,The Illuminati and Freemasons are often mixed ...,1
9196,They are secretive groups once made of wealthy...,1
...,...,...
1477,"Yes, some blood pressure change within an hour...",1
1478,A painless lump near the shoulder could be a b...,1
1479,"Acutret is isotretinoin, and it should only be...",1
1480,Yes. A pulse around 35 with dizziness and shor...,1


In [ ]:
# getting only human data from test, val, and train data
df_test = df_test.loc[df_test['label'] == 0]
df_val = df_val.loc[df_val['label']==0]
df_train = df_train.loc[df_train['label']==0]

In [ ]:
# creating the new dataset by unifying human and gpt5.4- mini dataset for each train, val, and test.
new_test = pd.concat([df_test,test_data])[['text', 'label']]
new_val = pd.concat([df_val,val_data])[['text', 'label']]
new_train = pd.concat([df_train,train_data])[['text', 'label']]

In [ ]:
# store the data as files
new_test.to_csv('/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/new_test.', index = False)
new_val.to_csv('/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/new_val.', index = False)
new_train.to_csv('/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/new_train.', index = False)

In [ ]:
# store the full data including questions and answers from GPT5.4 for further analysis
new_full_data = pd.concat([df_recent_1,df_recent_2,df_recent_3,df_recent_4,df_recent_5, df_recent_6, df_recent_7, df_recent_8, df_recent_9])[['question', 'gpt_5_4_mini']]
new_full_data.to_csv('/content/drive/MyDrive/Colab Notebooks/NLU_project/HC3_dataset/new_full_data.', index = False)